In [1]:
import os

In [2]:
%pwd

'c:\\Users\\user\\Desktop\\Data Science Projects\\End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\user\\Desktop\\Data Science Projects\\End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment'

In [5]:
import dagshub

In [6]:
dagshub.init(repo_owner='cerenn25', repo_name='End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment', mlflow=True)

Accessing as cerenn25

Initialized MLflow to track repo 
"cerenn25/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment"

Repository cerenn25/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment 
initialized!

In [7]:
import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

c:\Users\user\anaconda3\envs\kidney\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run thoughtful-rook-674 at: https://dagshub.com/cerenn25/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment.mlflow/#/experiments/0/runs/db24727845de4d978b2d210902afda37
🧪 View experiment at: https://dagshub.com/cerenn25/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment.mlflow/#/experiments/0


In [8]:
import tensorflow as tf

In [9]:
model = tf.keras.models.load_model("artifacts/training/model.keras")

ValueError: File not found: filepath=artifacts/training/model.keras. Please ensure the file is an accessible `.keras` zip file.

In [10]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [11]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [12]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            create_directories([self.config.atrifacts_root])

    def get_evaluation_config(self)  -> EvaluationConfig:
        eval_config = EvaluationConfig(
             path_of_model="artifacts/training/model.keras",
             training_data="artifacts/data_ingestion/",
             mlflow_uri="https://dagshub.com/cerenn25/End-to-End-Kidney-Disease-Deep-Learning-Classification-Project-MLflow-DVC-Deployment.mlflow/#/experiments/0/runs/c2ca414d9f9a4ce7862111b10d6f52e9",
             all_params=self.params,
             params_image_size=self.params.IMAGE_SIZE,
             params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [13]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [10]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size =self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self._valid_generator = valid_datagenerator.flow_from_directory(
            directorry = self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self._valid_generator)

NameError: name 'EvaluationConfig' is not defined